In [1]:
from preprocess.prepocess import preprocess
import pandas as pd


file_path = "syn_data/p0.8_mu0.2_1.csv"
node2id, num_nodes = preprocess(file_path=file_path)


50 nodes in the link stream


/Users/acw721/Desktop/research/linkstream/preprocess/prepocess.py:4: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  link_stream = pd.read_csv(


In [3]:
link_stream = pd.read_csv('preprocess/p0.8_mu0.2_1.csv')

In [4]:
from tgn.utils.my_data import get_data  
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

data = 'p0.8_mu0.2_1'

node_feat, edge_feat, full_data = get_data(data, "preprocess/p0.8_mu0.2_1.csv", node_embedding_method="ctdne")

max_idx = max(full_data.unique_nodes)

cannot find (./data/p0.8_mu0.2_1.npy), use zero-vector for edge feat (dim=16)...
Loading CTDNE node features: pretrain/p0.8_mu0.2_1.npy
The dataset has 469 interactions, involving 50 different nodes


In [54]:
from tgn.utils.utils import get_neighbor_finder
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.utils'])
from tgn.utils.utils import EarlyStopMonitor, get_neighbor_finder, MLP


ngh_finder = get_neighbor_finder(full_data, uniform=False, max_node_idx=max_idx)

In [22]:
from tgn.utils.my_data import get_data, compute_time_statistics, compute_time_statistics_undirected
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

mean_time_shift, std_time_shift= compute_time_statistics_undirected(full_data.sources, 
                                full_data.destinations, 
                                full_data.timestamps)

In [ ]:
import numpy as np
import torch
from sklearn.cluster import KMeans


def init_cluster_centers_from_tgn_embeddings(
    tgn,
    data,
    batch_size,
    n_neighbors,
    num_clusters,
    use_memory=True,
    max_samples=None,
):
    tgn.eval()

    z_list = []
    num_instance = len(data.sources)
    num_batch = (num_instance + batch_size - 1) // batch_size

    if use_memory and tgn.use_memory:
        tgn.memory.__init_memory__()

    with torch.no_grad():
        for k in range(num_batch):
            start_idx = k * batch_size
            end_idx = min(num_instance, start_idx + batch_size)

            sources_batch = data.sources[start_idx:end_idx]
            destinations_batch = data.destinations[start_idx:end_idx]
            edge_idxs_batch = data.edge_idxs[start_idx:end_idx]
            timestamps_batch = data.timestamps[start_idx:end_idx]

            negative_nodes = destinations_batch  # 占位，复用接口

            src_emb, dst_emb, _ = tgn.compute_temporal_embeddings(
                source_nodes=sources_batch,
                destination_nodes=destinations_batch,
                negative_nodes=negative_nodes,
                edge_times=timestamps_batch,
                edge_idxs=edge_idxs_batch,
                n_neighbors=n_neighbors
            )

            z_batch = torch.cat([src_emb, dst_emb], dim=0)   # [2B, D]
            z_list.append(z_batch.detach().cpu().numpy())

            if use_memory and tgn.use_memory:
                tgn.memory.detach_memory()

    Z = np.concatenate(z_list, axis=0)

    if max_samples is not None and len(Z) > max_samples:
        idx = np.random.choice(len(Z), size=max_samples, replace=False)
        Z = Z[idx]

    kmeans = KMeans(n_clusters=num_clusters, n_init=10, random_state=0)
    kmeans.fit(Z)

    centers = torch.tensor(
        kmeans.cluster_centers_,
        dtype=torch.float32,
        device=tgn.cluster_centers.device
    )

    with torch.no_grad():
        tgn.cluster_centers.copy_(centers)


def init_cluster_centers_from_saved_node_event_embeddings(
    tgn,
    num_clusters,
    src_npy_path="/Users/acw721/Desktop/research/linkstream/results/node_event_emb/full_src.npy",
    dst_npy_path="/Users/acw721/Desktop/research/linkstream/results/node_event_emb/full_dst.npy",
    max_samples=None,
    random_state=0,
):
    """

    参数
    ----
    tgn : your TGN model
    num_clusters : int
    src_npy_path : str
    dst_npy_path : str
    max_samples : Optional[int]
    random_state : int
    """
    src_emb = np.load(src_npy_path)   # [N, D]
    dst_emb = np.load(dst_npy_path)   # [N, D]

    if src_emb.ndim != 2 or dst_emb.ndim != 2:
        raise ValueError(f"Expected 2D arrays, got src={src_emb.shape}, dst={dst_emb.shape}")

    if src_emb.shape[1] != dst_emb.shape[1]:
        raise ValueError(
            f"src/dst feature dim mismatch: src={src_emb.shape}, dst={dst_emb.shape}"
        )

    X = np.concatenate([src_emb, dst_emb], axis=0)   # [2N, D]

    if max_samples is not None and len(X) > max_samples:
        idx = np.random.choice(len(X), size=max_samples, replace=False)
        X = X[idx]

    feat_dim = X.shape[1]
    model_dim = tgn.embedding_dimension

    if feat_dim != model_dim:
        raise ValueError(
            f"Saved embedding dim ({feat_dim}) != tgn.embedding_dimension ({model_dim}). "
            f"Need projection/PCA before initializing cluster centers."
        )

    kmeans = KMeans(
        n_clusters=num_clusters,
        n_init=10,
        random_state=random_state
    )
    kmeans.fit(X)

    centers = torch.tensor(
        kmeans.cluster_centers_,
        dtype=torch.float32,
        device=tgn.cluster_centers.device
    )

    with torch.no_grad():
        tgn.cluster_centers.copy_(centers)

    print("Initialized cluster centers from saved node-event embeddings")
    print("src_emb shape:", src_emb.shape)
    print("dst_emb shape:", dst_emb.shape)
    print("kmeans centers shape:", kmeans.cluster_centers_.shape)

In [177]:
import importlib
import tgn.model.my_tgn as my_tgn
importlib.reload(my_tgn)
TGN = my_tgn.TGN
from pathlib import Path


import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path
import torch

NUM_EPOCH = 100
BATCH_SIZE = 128
NUM_NEIGHBORS = 20
NUM_HEADS = 4
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.001
NODE_DIM = node_feat.shape[1]
TIME_DIM = 16
USE_MEMORY = False
MESSAGE_DIM = 16
#MEMORY_DIM = NODE_DIM
MEMORY_DIM = 16
num_communities = 5
device = 'cuda' if torch.cuda.is_available() else 'mps'
prefix = 'syn_net'
aggregator = 'mean'
memory_update_at_end = False
embedding_module = 'graph_attention' # graph_attention, graph_sum, time, identity
message_function = 'mlp'

Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)

tgn = TGN(
    neighbor_finder=ngh_finder,
    node_features=node_feat,
    edge_features=edge_feat,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift=mean_time_shift,
    std_time_shift=std_time_shift,
    use_destination_embedding_in_message=False,
    use_source_embedding_in_message=False,
    memory_update_at_start=not memory_update_at_end,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator,
    student_time_dim=32,
).to(device)


init_cluster_centers_from_saved_node_event_embeddings(
    tgn=tgn,
    num_clusters=num_communities,
    src_npy_path="/Users/acw721/Desktop/research/linkstream/pretrain/node_event_feat/src_feat.npy",
    dst_npy_path="/Users/acw721/Desktop/research/linkstream/pretrain/node_event_feat/dst_feat.npy",
    max_samples=None,
    random_state=0,
)

tgn.load_teacher_node_event_embeddings(
    src_npy_path="/Users/acw721/Desktop/research/linkstream/pretrain/node_event_feat/src_feat.npy",
    dst_npy_path="/Users/acw721/Desktop/research/linkstream/pretrain/node_event_feat/dst_feat.npy",
)
#tgn.cluster_centers.requires_grad_(False)
optimizer = torch.optim.Adam(tgn.parameters(), lr=LEARNING_RATE)

Initialized cluster centers from saved node-event embeddings
src_emb shape: (469, 96)
dst_emb shape: (469, 96)
kmeans centers shape: (5, 96)


In [178]:
sources_batch = full_data.sources[:32]
destinations_batch = full_data.destinations[:32]
timestamps_batch = full_data.timestamps[:32]
edge_idxs_batch = full_data.edge_idxs[:32]

with torch.no_grad():
    z_student = tgn.compute_node_event_embeddings_for_clustering(
        source_nodes=sources_batch,
        destination_nodes=destinations_batch,
        edge_times=timestamps_batch,
        edge_idxs=edge_idxs_batch,
        n_neighbors=NUM_NEIGHBORS
    )  # [2B, D]

    z_teacher = tgn.get_teacher_node_event_embeddings_for_batch(edge_idxs_batch)  # [2B, D]

    mse = ((z_student - z_teacher) ** 2).mean().item()
    cos = F.cosine_similarity(z_student, z_teacher, dim=1).mean().item()

    print("student vs teacher mse =", mse)
    print("student vs teacher mean cosine =", cos)

student vs teacher mse = 0.0
student vs teacher mean cosine = 1.0


In [168]:
from pathlib import Path
import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path
import torch


num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
print(f'num_batches: {num_batches}')

num_batches: 4


In [169]:
NUM_NEG = 10

def sample_negative_destinations(num_nodes, positive_destinations, num_neg):
  positive_destinations = np.asarray(positive_destinations, dtype=np.int64)
  batch_size = len(positive_destinations)
  neg_dst = np.random.randint(0, num_nodes, size=(batch_size, num_neg), dtype=np.int64)

  for i in range(batch_size):
    for j in range(num_neg):
      while neg_dst[i, j] == positive_destinations[i]:
        neg_dst[i, j] = np.random.randint(0, num_nodes)

  return neg_dst

In [171]:
import copy
import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from evaluation import dynamic_mi

def build_partition_from_arrays(
    sources,
    destinations,
    timestamps,
    source_labels,
    destination_labels,
    on_conflict="keep_last",
):
    part = {}

    def _assign(k, v):
        if k not in part:
            part[k] = v
            return
        if part[k] == v:
            return
        if on_conflict == "keep_first":
            return
        if on_conflict == "keep_last":
            part[k] = v
            return
        raise ValueError(f"Conflict on {k}: existing={part[k]} new={v}")

    for s, d, t, cs, cd in zip(
        sources, destinations, timestamps, source_labels, destination_labels
    ):
        _assign((int(s), int(t)), int(cs))
        _assign((int(d), int(t)), int(cd))

    return part


def evaluate_epoch_ami_in_memory(
    tgn,
    data,
    gt_partition,
    batch_size,
    n_neighbors,
    num_clusters,
    use_memory=True,
    average_method="arithmetic",
    normalisation="ami",
):
    tgn.eval()

    src_emb_list = []
    dst_emb_list = []

    all_sources = []
    all_destinations = []
    all_timestamps = []

    num_instance = len(data.sources)
    num_batch = (num_instance + batch_size - 1) // batch_size

    if use_memory and tgn.use_memory:
        tgn.memory.__init_memory__()
    tgn.reset_student_last_update()
    with torch.no_grad():
        for k in range(num_batch):
            start_idx = k * batch_size
            end_idx = min(num_instance, start_idx + batch_size)

            sources_batch = data.sources[start_idx:end_idx]
            destinations_batch = data.destinations[start_idx:end_idx]
            edge_idxs_batch = data.edge_idxs[start_idx:end_idx]
            timestamps_batch = data.timestamps[start_idx:end_idx]

            negative_nodes = destinations_batch

            
            """
            src_emb, dst_emb, _ = tgn.compute_temporal_embeddings(
                source_nodes=sources_batch,
                destination_nodes=destinations_batch,
                negative_nodes=negative_nodes,
                edge_times=timestamps_batch,
                edge_idxs=edge_idxs_batch,
                n_neighbors=n_neighbors
            )
            """
            
            z_batch = tgn.compute_node_event_embeddings_for_clustering(
                source_nodes=sources_batch,
                destination_nodes=destinations_batch,
                edge_times=timestamps_batch,
                edge_idxs=edge_idxs_batch,
                n_neighbors=n_neighbors
            )

            B = len(sources_batch)
            src_emb = z_batch[:B]
            dst_emb = z_batch[B:]
            # ===========================

            src_emb_list.append(src_emb.detach().cpu().numpy())
            dst_emb_list.append(dst_emb.detach().cpu().numpy())

            all_sources.append(np.asarray(sources_batch))
            all_destinations.append(np.asarray(destinations_batch))
            all_timestamps.append(np.asarray(timestamps_batch))

            if use_memory and tgn.use_memory:
                tgn.memory.detach_memory()

    src_emb_all = np.concatenate(src_emb_list, axis=0)
    dst_emb_all = np.concatenate(dst_emb_list, axis=0)

    sources_all = np.concatenate(all_sources, axis=0)
    destinations_all = np.concatenate(all_destinations, axis=0)
    timestamps_all = np.concatenate(all_timestamps, axis=0)

    # node-event clustering
    X = np.concatenate([src_emb_all, dst_emb_all], axis=0)
    labels = KMeans(
        n_clusters=num_clusters,
        n_init=10,
        random_state=0
    ).fit_predict(X)

    N = len(sources_all)
    src_labels = labels[:N]
    dst_labels = labels[N:]

    pred_partition = build_partition_from_arrays(
        sources=sources_all,
        destinations=destinations_all,
        timestamps=timestamps_all,
        source_labels=src_labels,
        destination_labels=dst_labels,
        on_conflict="keep_last",
    )

    score = dynamic_mi(
        gt=gt_partition,
        pred=pred_partition,
        average_method=average_method,
        normalisation=normalisation,
    )
    return score

In [172]:
import copy
from __future__ import annotations

from typing import Dict, Hashable, Tuple, Any, Optional, Literal
import pandas as pd

Node = Hashable
Time = Hashable
Key = Tuple[Node, Time]


def build_partition_from_csv(
    csv_path: str,
    *,
    source_col: str = "source",
    destination_col: str = "destination",
    timestamp_col: str = "timestamp",
    source_commu_col: str = "source_commu",
    destination_commu_col: str = "destination_commu",
    sep: str = ",",
    header: Optional[int] = "infer",
    skip_first_row: bool = False,
    dtype_source: Any = int,
    dtype_destination: Any = int,
    dtype_timestamp: Any = int,
    dtype_commu: Any = int,
    on_conflict: Literal["keep_first", "keep_last", "error"] = "keep_last",
    node2id: Optional[Dict[Any, int]] = None,
) -> Dict[Key, Any]:
    usecols = [source_col, destination_col, timestamp_col, source_commu_col, destination_commu_col]

    df = pd.read_csv(
        csv_path,
        sep=sep,
        header=header,
        usecols=usecols,
        skiprows=1 if skip_first_row else None,
    )

    df[source_col] = df[source_col].astype(dtype_source)
    df[destination_col] = df[destination_col].astype(dtype_destination)
    df[timestamp_col] = df[timestamp_col].astype(dtype_timestamp)
    df[source_commu_col] = df[source_commu_col].astype(dtype_commu)
    df[destination_commu_col] = df[destination_commu_col].astype(dtype_commu)

    part: Dict[Key, Any] = {}

    def _map_node(x):
        if node2id is None:
            return x
        if x not in node2id:
            raise KeyError(f"node {x} not found in node2id")
        return node2id[x]

    def _assign(k: Key, v: Any):
        if k not in part:
            part[k] = v
            return
        if part[k] == v:
            return
        if on_conflict == "keep_first":
            return
        if on_conflict == "keep_last":
            part[k] = v
            return
        raise ValueError(f"Conflict on {k}: existing={part[k]} new={v}")

    for s, d, t, cs, cd in df[
        [source_col, destination_col, timestamp_col, source_commu_col, destination_commu_col]
    ].itertuples(index=False, name=None):
        s = _map_node(s)
        d = _map_node(d)
        _assign((s, t), cs)
        _assign((d, t), cd)

    return part

# ====== AMI 评估相关初始化 ======
gt_partition = build_partition_from_csv(
    "syn_data/p0.8_mu0.2_1.csv",
    node2id=node2id
)

epoch_amis = []
train_losses = []

best_ami = -1.0
best_epoch = -1
best_state_dict = None

In [179]:
import numpy as np
import torch
from sklearn.cluster import KMeans

def evaluate_student_branch_ami(
    tgn,
    data,
    gt_partition,
    batch_size,
    n_neighbors,
    num_clusters,
    average_method="arithmetic",
    normalisation="ami",
):
    """
    直接评估 student 分支：
      compute_node_event_embeddings_for_clustering
      -> KMeans
      -> pred partition
      -> dynamic AMI
    """
    tgn.eval()
    tgn.reset_student_last_update()

    src_emb_list = []
    dst_emb_list = []

    all_sources = []
    all_destinations = []
    all_timestamps = []

    num_instance = len(data.sources)
    num_batch = (num_instance + batch_size - 1) // batch_size

    with torch.no_grad():
        for k in range(num_batch):
            start_idx = k * batch_size
            end_idx = min(num_instance, start_idx + batch_size)

            sources_batch = data.sources[start_idx:end_idx]
            destinations_batch = data.destinations[start_idx:end_idx]
            edge_idxs_batch = data.edge_idxs[start_idx:end_idx]
            timestamps_batch = data.timestamps[start_idx:end_idx]

            z_batch = tgn.compute_node_event_embeddings_for_clustering(
                source_nodes=sources_batch,
                destination_nodes=destinations_batch,
                edge_times=timestamps_batch,
                edge_idxs=edge_idxs_batch,
                n_neighbors=n_neighbors,
            )  # [2B, D]

            B = len(sources_batch)
            src_emb = z_batch[:B]
            dst_emb = z_batch[B:]

            src_emb_list.append(src_emb.detach().cpu().numpy())
            dst_emb_list.append(dst_emb.detach().cpu().numpy())

            all_sources.append(np.asarray(sources_batch))
            all_destinations.append(np.asarray(destinations_batch))
            all_timestamps.append(np.asarray(timestamps_batch))

    src_emb_all = np.concatenate(src_emb_list, axis=0)
    dst_emb_all = np.concatenate(dst_emb_list, axis=0)

    sources_all = np.concatenate(all_sources, axis=0)
    destinations_all = np.concatenate(all_destinations, axis=0)
    timestamps_all = np.concatenate(all_timestamps, axis=0)

    X = np.concatenate([src_emb_all, dst_emb_all], axis=0)

    labels = KMeans(
        n_clusters=num_clusters,
        n_init=10,
        random_state=0
    ).fit_predict(X)

    N = len(sources_all)
    src_labels = labels[:N]
    dst_labels = labels[N:]

    pred_partition = build_partition_from_arrays(
        sources=sources_all,
        destinations=destinations_all,
        timestamps=timestamps_all,
        source_labels=src_labels,
        destination_labels=dst_labels,
        on_conflict="keep_last",
    )

    score = dynamic_mi(
        gt=gt_partition,
        pred=pred_partition,
        average_method=average_method,
        normalisation=normalisation,
    )
    return score


student_ami_before_train = evaluate_student_branch_ami(
    tgn=tgn,
    data=full_data,
    gt_partition=gt_partition,
    batch_size=BATCH_SIZE,
    n_neighbors=NUM_NEIGHBORS,
    num_clusters=num_communities,
    average_method="arithmetic",
    normalisation="ami",
)

print("Student-branch AMI before training =", student_ami_before_train)

Student-branch AMI before training = 0.29092178177197675


In [ ]:
backprop_every = 1

for epoch in range(NUM_EPOCH):
  start_epoch = time.time()
  if USE_MEMORY:
    tgn.memory.__init_memory__()
  
  tgn.reset_student_last_update()
  tgn.set_neighbor_finder(ngh_finder)
  m_loss = []

  logger.info('start {} epoch'.format(epoch))

  for k in range(0, num_batches, backprop_every):
    loss = 0
    optimizer.zero_grad()

    for j in range(backprop_every):
      batch_idx = k + j
      if batch_idx >= num_batches:
        continue

      start_idx = batch_idx * BATCH_SIZE
      end_idx = min(num_instance, start_idx + BATCH_SIZE)

      sources_batch = full_data.sources[start_idx:end_idx]
      destinations_batch = full_data.destinations[start_idx:end_idx]
      edge_idxs_batch = full_data.edge_idxs[start_idx:end_idx]
      timestamps_batch = full_data.timestamps[start_idx:end_idx]

      neg_dst_batch = sample_negative_destinations(
        num_nodes=tgn.n_nodes,
        positive_destinations=destinations_batch,
        num_neg=NUM_NEG
      )

      tgn.train()
      mask_loss = tgn.compute_masked_destination_loss(
        source_nodes=sources_batch,
        destination_nodes=destinations_batch,
        negative_destination_nodes=neg_dst_batch,
        edge_times=timestamps_batch,
        edge_idxs=edge_idxs_batch,
        n_neighbors=NUM_NEIGHBORS
      )

      node_event_loss, q_batch = tgn.compute_node_event_cluster_loss(
        source_nodes=sources_batch,
        destination_nodes=destinations_batch,
        edge_times=timestamps_batch,
        edge_idxs=edge_idxs_batch,
        n_neighbors=NUM_NEIGHBORS,
        alpha=1.0
    )
      
    lambda_node = 1.0
    loss = 1.0 * mask_loss + lambda_node * node_event_loss

    loss /= backprop_every
    loss.backward()
    optimizer.step()
    m_loss.append(loss.item())

    if USE_MEMORY:
      tgn.memory.detach_memory()

  epoch_time = time.time() - start_epoch
  mean_train_loss = float(np.mean(m_loss))
  train_losses.append(mean_train_loss)

  logger.info('epoch: {} took {:.2f}s'.format(epoch, epoch_time))
  logger.info('Epoch mean mask loss: {}'.format(mean_train_loss))


  try:
    epoch_ami = evaluate_epoch_ami_in_memory(
      tgn=tgn,
      data=full_data,
      gt_partition=gt_partition,
      batch_size=BATCH_SIZE,
      n_neighbors=NUM_NEIGHBORS,
      num_clusters=num_communities,
      use_memory=USE_MEMORY,
      average_method="arithmetic",
      normalisation="ami",
    )
  except Exception as e:
    logger.exception(f"AMI evaluation failed at epoch {epoch}: {e}")
    epoch_ami = float("-inf")

  epoch_amis.append(epoch_ami)
  logger.info('Epoch AMI: {}'.format(epoch_ami))

  if epoch_ami > best_ami:
    best_ami = epoch_ami
    best_epoch = epoch
    best_state_dict = copy.deepcopy(tgn.state_dict())
    torch.save(best_state_dict, "saved_models/best_tgn_mask_by_ami.pth")
    logger.info(f"New best model saved at epoch {epoch}, AMI={epoch_ami:.6f}")

INFO:root:start 0 epoch
INFO:root:epoch: 0 took 0.68s
INFO:root:Epoch mean mask loss: 7.1649090051651
INFO:root:Epoch AMI: 0.29200795417178316
INFO:root:New best model saved at epoch 0, AMI=0.292008
INFO:root:start 1 epoch
INFO:root:epoch: 1 took 0.23s
INFO:root:Epoch mean mask loss: 5.849570870399475
INFO:root:Epoch AMI: 0.29227298536190427
INFO:root:New best model saved at epoch 1, AMI=0.292273
INFO:root:start 2 epoch
INFO:root:epoch: 2 took 0.23s
INFO:root:Epoch mean mask loss: 4.543922424316406
INFO:root:Epoch AMI: 0.2929841174310597
INFO:root:New best model saved at epoch 2, AMI=0.292984
INFO:root:start 3 epoch
INFO:root:epoch: 3 took 0.23s
INFO:root:Epoch mean mask loss: 3.8412408232688904
INFO:root:Epoch AMI: 0.2942108833888224
INFO:root:New best model saved at epoch 3, AMI=0.294211
INFO:root:start 4 epoch
INFO:root:epoch: 4 took 0.24s
INFO:root:Epoch mean mask loss: 3.5817049741744995
INFO:root:Epoch AMI: 0.29421088338882234
INFO:root:start 5 epoch
INFO:root:epoch: 5 took 0.24s

KeyboardInterrupt: 

In [68]:
import os
import numpy as np
import pandas as pd
import torch


def export_node_event_embeddings_with_raw_id(
    tgn,
    data,
    batch_size,
    out_prefix,
    n_neighbors,
    id2node,
    use_memory=True
):
    """
    导出每条事件两端节点在该事件时刻的 embedding，
    并把内部节点 id 映射回原始节点 id。

    输出：
      1. {out_prefix}_src.npy
      2. {out_prefix}_dst.npy
      3. {out_prefix}_meta.csv   (source/destination 已映射回原始 id)
    """
    tgn.eval()

    src_emb_list = []
    dst_emb_list = []
    meta_list = []

    num_instance = len(data.sources)
    num_batch = (num_instance + batch_size - 1) // batch_size

    if use_memory and tgn.use_memory:
        tgn.memory.__init_memory__()

    with torch.no_grad():
        for k in range(num_batch):
            start_idx = k * batch_size
            end_idx = min(num_instance, start_idx + batch_size)

            sources_batch = data.sources[start_idx:end_idx]
            destinations_batch = data.destinations[start_idx:end_idx]
            timestamps_batch = data.timestamps[start_idx:end_idx]
            edge_idxs_batch = data.edge_idxs[start_idx:end_idx]

            negative_nodes = destinations_batch

            src_emb, dst_emb, _ = tgn.compute_temporal_embeddings(
                source_nodes=sources_batch,
                destination_nodes=destinations_batch,
                negative_nodes=negative_nodes,
                edge_times=timestamps_batch,
                edge_idxs=edge_idxs_batch,
                n_neighbors=n_neighbors
            )

            src_emb_list.append(src_emb.detach().cpu().numpy())
            dst_emb_list.append(dst_emb.detach().cpu().numpy())

            raw_sources = [id2node[int(x)] for x in sources_batch]
            raw_destinations = [id2node[int(x)] for x in destinations_batch]

            meta_list.append(pd.DataFrame({
                "source": raw_sources,
                "destination": raw_destinations,
                "timestamp": timestamps_batch,
                "edge_idx": edge_idxs_batch,
            }))

            if use_memory and tgn.use_memory:
                tgn.memory.detach_memory()

    src_emb_all = np.concatenate(src_emb_list, axis=0)
    dst_emb_all = np.concatenate(dst_emb_list, axis=0)
    meta_df = pd.concat(meta_list, axis=0, ignore_index=True)

    os.makedirs(os.path.dirname(out_prefix), exist_ok=True)
    np.save(out_prefix + "_src.npy", src_emb_all)
    np.save(out_prefix + "_dst.npy", dst_emb_all)
    meta_df.to_csv(out_prefix + "_meta.csv", index=False)

    print("saved:", out_prefix + "_src.npy", src_emb_all.shape)
    print("saved:", out_prefix + "_dst.npy", dst_emb_all.shape)
    print("saved:", out_prefix + "_meta.csv", len(meta_df))

    return src_emb_all, dst_emb_all, meta_df


id2node = {idx: node for node, idx in node2id.items()}

src_emb, dst_emb, meta_df = export_node_event_embeddings_with_raw_id(
    tgn=tgn,
    data=full_data,
    batch_size=BATCH_SIZE,
    out_prefix="results/node_event_emb/full",
    n_neighbors=NUM_NEIGHBORS,
    id2node=id2node,
    use_memory=USE_MEMORY
)

saved: results/node_event_emb/full_src.npy (469, 64)
saved: results/node_event_emb/full_dst.npy (469, 64)
saved: results/node_event_emb/full_meta.csv 469
